In [21]:
## Imports ##
import pandas as pd
import re

### Loading Necessary CSVs

In [22]:
## All sheets loaded as Pandas DataFrames, then converted to dicts ##
# Gis sheet loading
g = pd.read_csv("gis_sheet.csv")
g["uri"] = g["uri"].astype("string")

# OC sheet loading
oc = pd.read_csv("trench-data copy.csv")

# Only keep necessary info
keys_to_drop = ["Citation URI", "Project", "Project URI", "Context", "Context [2]", "Context [3]", "Context URI", "Latitude (WGS 84)", "Longitude (WGS 84)", "Early BCE/CE", "Late BCE/CE", "Item Category", "icon", "Thumbnail", "Published Date", "Updated Date"]
oc = oc.drop(columns=keys_to_drop)
# Inserting new rows for data
oc.insert(4, "Trench Abbreviation", "")
oc.insert(5, "Trench Number", "")
oc.insert(6, "Year", 0)

# U sheet creation (starts blank), populated during operation. Won't be added to until all possible URIs matched
u = {
    "URI" : [],
    "Trench Area" : [],
    "Trench #" : []
}

### Adjusting OC Data to Match D

In [23]:
# Going thru OC and changing site area to abbreviated to make sheet matching w/ g easier
# Also separating trench # from trench area
for i in range(len(oc)):
    ## Abbreviating site area from Context [4] ##
    # Split area into separate words
    curr_area = oc["Context [4]"][i]
    words = curr_area.split()
    new_abbrev = ""

    # First letter of each word
    for w in words:
        new_abbrev += w[0]

    oc.loc[i, "Trench Abbreviation"] = new_abbrev


    ## Extracting trench number out of Context [5] field for comparisons ##
    curr_trench = oc.loc[i, "Context [5]"]
    # Locate first digit (almost always is the trench number)
    begin = len(curr_trench)
    for letter, c in enumerate(curr_trench):
        if c.isdigit():
            begin = letter
            break

    # Locate where that number ends
    end = str.find(curr_trench, " ", begin)

    # If it's not found, same as start so it's empty
    if end == -1:
        end = len(curr_trench)

    oc.loc[i, "Trench Number"] = curr_trench[begin:end] # Assign value

    # Using regex (?<!\d)\d{4}(?!\d) to filter item label to find year (thanks Google)
    year_obj = re.search(r"(?<!\d)\d{4}(?!\d)", oc["Item Label"][i])
    try:    # Some don't have year in their label, which causes an error. Itokay
        year = int(year_obj[0])
    except:
        year = 0
    oc.loc[i, "Year"] = year

### Main Iteration, Populating URIs

In [24]:
oc = pd.DataFrame(oc)
for g_i in range(len(g)):
    #print("main loop") #for debug
    # Define what we're currently searching for
    target_area = g["trench_area"][g_i]
    target_trench = g["trench_number"][g_i]
    target_year = g["year"][g_i]

    if g_i > 0:
        # We've already checked this section - skip
        if g.loc[g_i, "polygon_id"] == g.loc[g_i - 1,"polygon_id"]:
            #print("skip") #for debug
            continue

    # Loop through oc to look for corresponding URI
    for oc_i in oc.index:
        # If both fields match, bingo
        #print("looking for match") #for debug

        if oc.loc[oc_i, "Trench Abbreviation"] == target_area:  # Matching trench area
            #print("abbrev match") #for debug

            if oc.loc[oc_i, "Trench Number"] == target_trench:  # Matching trench #
                if oc.loc[oc_i, "Year"] == target_year or oc.loc[oc_i, "Year"] == 0:    # Matching year
                    #print("found match") #for debug

                    # Now we need to go through all the fields with the same polygon_id since they're the same trench
                    this_polyid = g["polygon_id"][g_i]  # ID we are currently populating
                    curr_idx = g_i                      # Index (to ensure we don't go past this polyid)
                    polyid = g["polygon_id"][curr_idx]  # PolyID of current index

                    while polyid == this_polyid and curr_idx < len(g):
                        #print(f"Populating PolygonID #{polyid}") #for debug
                        # Set correct URI
                        g.loc[curr_idx, "uri"] = oc["URI"][oc_i]

                        # Increment
                        curr_idx += 1
                        if curr_idx < len(g):
                            polyid = g["polygon_id"][curr_idx]

                    # Remove that row's data so no duplicates
                    oc = oc.drop(index=oc_i)    # This is a dumb way of doing it but ummmm it's okay!!
                    break

### For Debugging

In [25]:
# print(oc)
# print(g)

### Exporting Results as CSV

In [26]:
# Trenches spreadsheet for QGIS (inserted URIs)
g.to_csv("TrenchesWithURI.csv")

In [27]:
# URIs that did not find a match (will have to be inserted manually)
oc.to_csv("UnmatchedURIs.csv")